# Retail Data ELT Lab — SQLite and Python

**Student starter notebook**

Use this notebook with the lab instructions to load the instructor-provided CSV files into SQLite, inspect their schemas, create staging views, run data-quality checks, and answer business questions.

**Tools:** Python, pandas, SQLite, Google Colab. No BigQuery or dbt setup is required.

Before starting, put `customers.csv`, `products.csv`, `orders_raw.csv`, and `orders_clean.csv` in a folder in your Google Drive, or use the direct-upload option in Step 1. Run cells from top to bottom. Use actual column names from your files; examples below are templates.

In [6]:
import sqlite3
from pathlib import Path
import pandas as pd

print("Python and SQLite are ready.")
print("SQLite version:", sqlite3.sqlite_version)
print("pandas version:", pd.__version__)

Python and SQLite are ready.
SQLite version: 3.45.1
pandas version: 2.2.3


## 1. Make the CSV files available

Choose **one** option: Google Drive (recommended) or direct upload. If using Drive, copy the four CSV files into a folder you own, for example `MyDrive/Retail_ELT_Lab`.

In [3]:
# Search for customers.csv in the workspace
import os
from pathlib import Path

for root, dirs, files in os.walk('/content'):
    for file in files:
        if 'customer' in file.lower():
            print("Found customer-related file:", os.path.join(root, file))

# Let's also set the DATA_DIR correctly to where the files are.
DATA_DIR = Path('/content')
required_files = ['customers.csv', 'products.csv', 'orders_raw.csv', 'orders_clean.csv']
for filename in required_files:
    file_path = DATA_DIR / filename
    print(f"{filename}: {'FOUND' if file_path.exists() else 'NOT FOUND'}")

Found customer-related file: /content/customers.csv
customers.csv: FOUND
products.csv: FOUND
orders_raw.csv: FOUND
orders_clean.csv: FOUND


In [ ]:
# OPTION B: Upload files directly from your computer.
# Run this cell ONLY if you are not using Google Drive.
# from google.colab import files
# uploaded = files.upload()
# DATA_DIR = Path('/content')
#
# After upload, verify that all four CSV files are in /content.

## 2. Read and inspect the CSV files

Run this after all four files show as found (or after direct upload). Review the columns and sample records before writing SQL.

In [7]:
customers_df = pd.read_csv(DATA_DIR / 'customers.csv')
products_df = pd.read_csv(DATA_DIR / 'products.csv')
orders_raw_df = pd.read_csv(DATA_DIR / 'orders_raw.csv')
orders_clean_df = pd.read_csv(DATA_DIR / 'orders_clean.csv')

dataframes = {
    'customers': customers_df,
    'products': products_df,
    'orders_raw': orders_raw_df,
    'orders_clean': orders_clean_df
}

for name, df in dataframes.items():
    print(f"\n--- {name} ---")
    print("Shape (rows, columns):", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())


--- customers ---
Shape (rows, columns): (500, 7)
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'city', 'state', 'signup_date']


,customer_id,first_name,last_name,email,city,state,signup_date
0,C0001,James,Gupta,james.gupta1@example.com,Dallas,TX,2023-07-18
1,C0002,Krishna,Khan,krishna.khan2@example.com,Houston,TX,2022-07-29
2,C0003,Maria,Davis,maria.davis3@example.com,Austin,TX,2024-05-14
3,C0004,Aisha,Sharma,aisha.sharma4@example.com,Austin,TX,2023-03-24
4,C0005,Krishna,Brown,krishna.brown5@example.com,Boston,MA,2022-02-24



--- products ---
Shape (rows, columns): (20, 4)
Columns: ['product_id', 'product_name', 'category', 'price']


,product_id,product_name,category,price
0,P001,Wireless Headphones,Electronics,89.99
1,P002,Bluetooth Speaker,Electronics,49.99
2,P003,Laptop Stand,Electronics,34.99
3,P004,USB-C Hub,Electronics,29.99
4,P005,Smart Watch,Electronics,199.99



--- orders_raw ---
Shape (rows, columns): (5100, 6)
Columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']


,order_id,customer_id,product_id,quantity,order_date,status
0,O02814,C0472,P011,4,2025-07-15,pending
1,O04017,C0492,P020,5,2025-04-30,pending
2,O03993,C0065,P001,5,2025-01-10,delivered
3,O04314,C0107,P011,1,2025-07-03,delivered
4,O01732,C0175,P019,2,2024-04-28,shipped



--- orders_clean ---
Shape (rows, columns): (5000, 6)
Columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']


,order_id,customer_id,product_id,quantity,order_date,status
0,O00001,C0397,P009,4,2025-09-18,delivered
1,O00002,C0147,P016,2,2024-03-16,delivered
2,O00003,C0232,P006,4,2025-11-29,cancelled
3,O00004,C0496,P011,3,2025-07-17,shipped
4,O00005,C0278,P010,3,2025-12-29,pending


In [8]:
for name, df in dataframes.items():
    print(f"\n--- {name}: data types ---")
    display(df.dtypes.to_frame('data_type'))
    print("Missing values by column:")
    display(df.isna().sum().to_frame('missing_count'))


--- customers: data types ---


,data_type
customer_id,object
first_name,object
last_name,object
email,object
city,object
state,object
signup_date,object


Missing values by column:


,missing_count
customer_id,0
first_name,0
last_name,0
email,0
city,0
state,0
signup_date,0



--- products: data types ---


,data_type
product_id,object
product_name,object
category,object
price,float64


... (6 outputs left out)


--- orders_clean: data types ---


,data_type
order_id,object
customer_id,object
product_id,object
quantity,int64
order_date,object
status,object


Missing values by column:


,missing_count
order_id,0
customer_id,0
product_id,0
quantity,0
order_date,0
status,0


## 3. Create a SQLite database and load the raw tables

Load `customers.csv`, `products.csv`, and `orders_raw.csv` into SQLite. Keep `orders_clean.csv` as a reference file for later comparison; do not use it to replace the raw orders table.

In [9]:
DB_PATH = Path('/content/retail_lab.db')
conn = sqlite3.connect(DB_PATH)

raw_dataframes = {
    'customers': customers_df,
    'products': products_df,
    'orders_raw': orders_raw_df
}

for table_name, df in raw_dataframes.items():
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Loaded {len(df):,} rows into {table_name}")

print("SQLite database:", DB_PATH)

Loaded 500 rows into customers
Loaded 20 rows into products
Loaded 5,100 rows into orders_raw
SQLite database: /content/retail_lab.db


In [10]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)
display(tables)

for table_name in ['customers', 'products', 'orders_raw']:
    count = pd.read_sql_query(
        f'SELECT COUNT(*) AS row_count FROM "{table_name}"',
        conn
    ).iloc[0]['row_count']
    print(f"{table_name}: {count:,} rows")

,name
0,customers
1,orders_raw
2,products


customers: 500 rows
products: 20 rows
orders_raw: 5,100 rows


In [11]:
display(pd.read_sql_query('SELECT * FROM orders_raw LIMIT 10;', conn))

,order_id,customer_id,product_id,quantity,order_date,status
0,O02814,C0472,P011,4,2025-07-15,pending
1,O04017,C0492,P020,5,2025-04-30,pending
2,O03993,C0065,P001,5,2025-01-10,delivered
3,O04314,C0107,P011,1,2025-07-03,delivered
4,O01732,C0175,P019,2,2024-04-28,shipped
5,O03546,C0343,P007,5,2024-01-27,delivered
6,O02403,C0189,P020,4,2025-07-28,pending
7,O04813,C0455,P015,3,2024-11-04,shipped
8,O01179,C0424,P010,3,2025-12-05,delivered
9,O00202,C0103,P019,5,2025-07-17,delivered


## 4. Inspect SQLite schemas and table grain

Review the exact field names and data types. Decide what one row represents in each table (for example, one customer, one product, one order, or one order line).

In [12]:
for table_name in ['customers', 'products', 'orders_raw']:
    print(f"\n--- {table_name} schema ---")
    display(pd.read_sql_query(f'PRAGMA table_info("{table_name}");', conn))


--- customers schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,customer_id,TEXT,0,None,0
1,1,first_name,TEXT,0,None,0
2,2,last_name,TEXT,0,None,0
3,3,email,TEXT,0,None,0
4,4,city,TEXT,0,None,0
5,5,state,TEXT,0,None,0
6,6,signup_date,TEXT,0,None,0



--- products schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,product_id,TEXT,0,None,0
1,1,product_name,TEXT,0,None,0
2,2,category,TEXT,0,None,0
3,3,price,REAL,0,None,0



--- orders_raw schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,customer_id,TEXT,0,None,0
2,2,product_id,TEXT,0,None,0
3,3,quantity,INTEGER,0,None,0
4,4,order_date,TEXT,0,None,0
5,5,status,TEXT,0,None,0


## 5. Create staging views with SQL

These starter views select all columns so they can run before you know the final field names. Improve them by selecting fields needed for analysis, standardizing text where appropriate, and documenting quality rules. If you edit a view definition, rerun its cell; each cell drops and recreates the view.

In [13]:
conn.execute("DROP VIEW IF EXISTS stg_customers;")
conn.execute("""
CREATE VIEW stg_customers AS
SELECT
    TRIM(customer_id) AS customer_id,
    TRIM(first_name) AS first_name,
    TRIM(last_name) AS last_name,
    TRIM(LOWER(email)) AS email,
    TRIM(city) AS city,
    TRIM(UPPER(state)) AS state,
    DATE(signup_date) AS signup_date
FROM customers;
""")
conn.commit()
display(pd.read_sql_query('SELECT * FROM stg_customers LIMIT 5;', conn))

,customer_id,first_name,last_name,email,city,state,signup_date
0,C0001,James,Gupta,james.gupta1@example.com,Dallas,TX,2023-07-18
1,C0002,Krishna,Khan,krishna.khan2@example.com,Houston,TX,2022-07-29
2,C0003,Maria,Davis,maria.davis3@example.com,Austin,TX,2024-05-14
3,C0004,Aisha,Sharma,aisha.sharma4@example.com,Austin,TX,2023-03-24
4,C0005,Krishna,Brown,krishna.brown5@example.com,Boston,MA,2022-02-24


In [14]:
conn.execute("DROP VIEW IF EXISTS stg_products;")
conn.execute("""
CREATE VIEW stg_products AS
SELECT
    TRIM(product_id) AS product_id,
    TRIM(product_name) AS product_name,
    TRIM(category) AS category,
    CAST(price AS REAL) AS price
FROM products;
""")
conn.commit()
display(pd.read_sql_query('SELECT * FROM stg_products LIMIT 5;', conn))

,product_id,product_name,category,price
0,P001,Wireless Headphones,Electronics,89.99
1,P002,Bluetooth Speaker,Electronics,49.99
2,P003,Laptop Stand,Electronics,34.99
3,P004,USB-C Hub,Electronics,29.99
4,P005,Smart Watch,Electronics,199.99


In [15]:
conn.execute("DROP VIEW IF EXISTS stg_orders;")
conn.execute("""
CREATE VIEW stg_orders AS
SELECT
    TRIM(o.order_id) AS order_id,
    TRIM(o.customer_id) AS customer_id,
    TRIM(o.product_id) AS product_id,
    CAST(o.quantity AS INTEGER) AS quantity,
    DATE(o.order_date) AS order_date,
    TRIM(LOWER(o.status)) AS status,
    ROUND(CAST(o.quantity AS INTEGER) * CAST(p.price AS REAL), 2) AS line_total
FROM orders_raw o
LEFT JOIN products p ON o.product_id = p.product_id;
""")
conn.commit()
display(pd.read_sql_query('SELECT * FROM stg_orders LIMIT 5;', conn))

,order_id,customer_id,product_id,quantity,order_date,status,line_total
0,O02814,C0472,P011,4,2025-07-15,pending,319.96
1,O04017,C0492,P020,5,2025-04-30,pending,84.95
2,O03993,C0065,P001,5,2025-01-10,delivered,449.95
3,O04314,C0107,P011,1,2025-07-03,delivered,79.99
4,O01732,C0175,P019,2,2024-04-28,shipped,199.98


### Example transformation pattern — adapt to your schema

This is only an example; it will not run unless the fields exist in your table.

```sql
SELECT
    order_id,
    product_id,
    quantity,
    unit_price,
    CAST(quantity AS REAL) * CAST(unit_price AS REAL) AS line_total
FROM orders_raw;
```

SQLite casts can behave unexpectedly for invalid text values. Inspect the raw values and document conversion rules before relying on calculated values.

## 6. Data-quality checks

The examples below assume `customer_id` exists and that the customers table should have one row per customer. Change the field name or rule to match your actual schema.

In [16]:
# Step 6: Missing or blank customer IDs
print('--- Data Quality Check 1: Missing or blank customer IDs ---')
display(pd.read_sql_query('SELECT COUNT(*) AS missing_customer_ids FROM stg_customers WHERE customer_id IS NULL OR TRIM(CAST(customer_id AS TEXT)) = "";', conn))

print('\n--- Data Quality Check 2: Missing or blank product IDs ---')
display(pd.read_sql_query('SELECT COUNT(*) AS missing_product_ids FROM stg_products WHERE product_id IS NULL OR TRIM(CAST(product_id AS TEXT)) = "";', conn))

print('\n--- Data Quality Check 3: Missing or blank order IDs ---')
display(pd.read_sql_query('SELECT COUNT(*) AS missing_order_ids FROM stg_orders WHERE order_id IS NULL OR TRIM(CAST(order_id AS TEXT)) = "";', conn))

--- Data Quality Check 1: Missing or blank customer IDs ---


,missing_customer_ids
0,0



--- Data Quality Check 2: Missing or blank product IDs ---


,missing_product_ids
0,0



--- Data Quality Check 3: Missing or blank order IDs ---


,missing_order_ids
0,0


In [17]:
# Step 6: Duplicate keys and Referential Integrity
print('--- Data Quality Check 4: Duplicate customer keys ---')
display(pd.read_sql_query('SELECT customer_id, COUNT(*) AS records FROM stg_customers GROUP BY customer_id HAVING COUNT(*) > 1;', conn))

print('\n--- Data Quality Check 5: Duplicate product keys ---')
display(pd.read_sql_query('SELECT product_id, COUNT(*) AS records FROM stg_products GROUP BY product_id HAVING COUNT(*) > 1;', conn))

print('\n--- Data Quality Check 6: Orphaned orders (Referential Integrity Check) ---')
display(pd.read_sql_query('SELECT COUNT(*) AS orphaned_orders_count FROM stg_orders WHERE customer_id NOT IN (SELECT customer_id FROM stg_customers) OR product_id NOT IN (SELECT product_id FROM stg_products);', conn))

--- Data Quality Check 4: Duplicate customer keys ---


,customer_id,records



--- Data Quality Check 5: Duplicate product keys ---


,product_id,records



--- Data Quality Check 6: Orphaned orders (Referential Integrity Check) ---


,orphaned_orders_count
0,0


## 7. Analytical SQL — answer business questions

Use your actual column names. First inspect `stg_orders`, `stg_customers`, and `stg_products` and identify:
- the order identifier and customer identifier
- the product identifier (if each row represents an order line)
- the order date
- the quantity, unit price, discount, and line total (where available)
- any order status, region, or category fields

**Important:** The queries below are templates, not guaranteed to run unchanged. Replace placeholder column names with fields in your files. If a field is not available, explain that limitation and choose a suitable alternative. Run at least four queries and write 1–3 sentences interpreting each result.


In [18]:
# Step 7: Analytical SQL — Query A: overall performance
print('--- Query A: Overall performance ---')
display(pd.read_sql_query('SELECT COUNT(*) AS order_line_count, COUNT(DISTINCT order_id) AS distinct_orders, SUM(quantity) AS units_sold, ROUND(SUM(line_total), 2) AS total_sales, ROUND(AVG(line_total), 2) AS average_line_value FROM stg_orders;', conn))

--- Query A: Overall performance ---


,order_line_count,distinct_orders,units_sold,total_sales,average_line_value
0,5100,5000,14947,1113264.53,218.29


In [19]:
# Step 7: Queries B, C, D, and E
print('--- Query B: Product performance ---')
display(pd.read_sql_query('SELECT p.product_name, SUM(o.quantity) AS units_sold, ROUND(SUM(o.line_total), 2) AS product_sales, COUNT(DISTINCT o.order_id) AS orders_including_product FROM stg_orders AS o JOIN stg_products AS p ON o.product_id = p.product_id GROUP BY p.product_name ORDER BY product_sales DESC LIMIT 10;', conn))

print('\n--- Query C: Customer-level performance ---')
display(pd.read_sql_query('SELECT c.customer_id, COUNT(DISTINCT o.order_id) AS order_count, ROUND(SUM(o.line_total), 2) AS customer_sales, ROUND(AVG(o.line_total), 2) AS average_line_value FROM stg_customers AS c LEFT JOIN stg_orders AS o ON c.customer_id = o.customer_id GROUP BY c.customer_id ORDER BY customer_sales DESC LIMIT 10;', conn))

print('\n--- Query D: Monthly sales trend ---')
display(pd.read_sql_query('SELECT substr(order_date, 1, 7) AS order_month, COUNT(DISTINCT order_id) AS order_count, ROUND(SUM(line_total), 2) AS monthly_sales FROM stg_orders WHERE order_date IS NOT NULL GROUP BY substr(order_date, 1, 7) ORDER BY order_month;', conn))

print('\n--- Query E: Duplicate order identifiers (inspecting table grain) ---')
display(pd.read_sql_query('SELECT order_id, COUNT(*) AS rows_per_order FROM stg_orders GROUP BY order_id HAVING COUNT(*) > 1 ORDER BY rows_per_order DESC LIMIT 10;', conn))

--- Query B: Product performance ---


,product_name,units_sold,product_sales,orders_including_product
0,Smart Watch,926,185190.74,298
1,Cookware Set,648,103673.52,225
2,Winter Jacket,686,102893.14,229
3,Running Shoes,746,96972.54,255
4,Mechanical Keyboard,732,87832.68,251
5,Tennis Racket,804,80391.96,280
6,Wireless Headphones,760,68392.40,252
7,Coffee Maker,854,68311.46,282
8,Dumbbell Pair,739,51722.61,261
9,Denim Jeans,748,44872.52,238



--- Query C: Customer-level performance ---


,customer_id,order_count,customer_sales,average_line_value
0,C0152,13,6777.42,451.83
1,C0103,13,5328.54,380.61
2,C0232,18,5269.37,292.74
3,C0301,17,5219.44,307.03
4,C0192,14,4899.47,349.96
5,C0403,18,4879.38,271.08
6,C0104,14,4842.55,345.90
7,C0497,15,4814.48,320.97
8,C0384,13,4794.62,368.82
9,C0198,12,4779.56,398.30



--- Query D: Monthly sales trend ---


,order_month,order_count,monthly_sales
0,2024-01,210,52877.46
1,2024-02,197,41020.31
2,2024-03,196,46057.72
3,2024-04,212,43472.88
4,2024-05,215,46684.57
5,2024-06,203,47876.06
6,2024-07,204,48946.04
7,2024-08,199,44149.94
8,2024-09,211,44756.46
9,2024-10,203,39988.17



--- Query E: Duplicate order identifiers (inspecting table grain) ---


,order_id,rows_per_order
0,O04994,2
1,O04974,2
2,O04949,2
3,O04864,2
4,O04771,2
5,O04770,2
6,O04691,2
7,O04673,2
8,O04666,2
9,O04618,2


## 8. Reconcile your results with `orders_clean.csv`

Treat `orders_clean.csv` as a reference dataset. Do not assume it has the same grain as `orders_raw` or your staging view. First inspect the fields and determine whether each row represents an order or an order line.

Complete these checks:
1. Compare row counts and column names.
2. Check for missing and duplicate business keys in the reference file.
3. Compare totals only when the two datasets use the same grain and the same definition of sales.
4. If both datasets share a reliable key, compare key coverage and identify records present in only one dataset.
5. If you compare numeric values, report the absolute difference and use a sensible rounding tolerance.
6. Explain at least two differences or data-quality issues you found. If the files cannot be compared directly, explain why and define the transformation needed to make them comparable.


In [20]:
# Step 8: Reconcile results with orders_clean.csv
print('Reference dataset shape:', orders_clean_df.shape)
print('Reference columns:', orders_clean_df.columns.tolist())

reference_key = 'order_id'
print('\nMissing reference keys:', int(orders_clean_df[reference_key].isna().sum()))
print('Duplicate reference keys:', int(orders_clean_df[reference_key].duplicated().sum()))

transformed_df = pd.read_sql_query('SELECT order_id, ROUND(SUM(line_total), 2) AS calculated_sales FROM stg_orders GROUP BY order_id;', conn)
comparison = transformed_df.merge(orders_clean_df[[reference_key, 'quantity']].rename(columns={reference_key: 'order_id', 'quantity': 'reference_units'}), on='order_id', how='outer', indicator=True)
print('\nMerge results:')
print(comparison['_merge'].value_counts())

Reference dataset shape: (5000, 6)
Reference columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']

Missing reference keys: 0
Duplicate reference keys: 0

Merge results:
_merge
both          5000
left_only        0
right_only       0
Name: count, dtype: int64


## 9. Save your work

Save the notebook to Drive using **File → Save a copy in Drive**. Download the SQLite database before the Colab runtime ends.

In [ ]:
conn.commit()
# Run this when you have finished using the database connection.
# conn.close()

from google.colab import files
files.download('/content/retail_lab.db')

## Student checklist

- [ ] Inspected all four CSV files.
- [ ] Loaded three raw tables into SQLite.
- [ ] Documented actual schemas and table grain.
- [ ] Created and improved all three staging views using actual field names.
- [ ] Ran data-quality checks and explained decisions.
- [ ] Answered at least three business questions with SQL.
- [ ] Compared results with `orders_clean.csv`.
- [ ] Saved the notebook and downloaded `retail_lab.db`.